# Copy-Paste 품질 개선과 검증

이 Notebook은 프로젝트에서 실제 수행하고 로컬 manifest/validation으로 확인된 Copy-Paste 개선 과정을 정리한다. 새로운 과거 실험 결과를 만들어내지 않으며, 원본 이미지나 Annotation을 수정하지 않는다. 최종 확인된 버전 체계는 **CP v1 → CP v2 → CP v3(CP63) → CP v4(CP126) → CP v5(CP500 final)**이다.

실제 데이터는 Git에 포함하지 않는다. 아래 설정 셀에서 각 데이터셋 경로를 지정하면 manifest 확인과 공통 품질 검증을 다시 실행할 수 있다.

## 1. 실제 프로젝트 진행 흐름

1. CP v1: 투명 객체와 실제 Train 배경을 이용한 초기 합성 결과이며 Drive에는 압축파일로 보관한다.
2. CP v2: 초기 검수 내용을 반영한 후속 실험 결과이며 Drive에는 압축파일로 보관한다. v1/v2 육안 및 자동 검수에서 객체 겹침, 최소 간격 부족, 객체 크기 이상, 경계 이탈 가능성을 확인했다.
3. CP v3 (CP63): 동일 category의 실제 Train bbox shape와 호환되는 객체만 `use`로 선별하고 deterministic isotropic scaling을 적용했다. 확인된 결과는 **63 images / 207 objects**이다.
4. CP v4 (CP126): CP63을 보존하면서 신규 63장을 추가하고 basename, ID, bbox, overlap 및 중복을 검증했다. 확인된 결과는 **126 images / 414 objects**이다.
5. CP v5 (CP500 final): 겹침·20 px 간격·경계·클래스별 크기 이상 및 image/annotation pair를 최종 검증했다. 확인된 결과는 **500 images / 500 annotations / 1,643 objects**이다.

위 수치는 기존 `generation_manifest.csv` 및 `validation_summary.json`에서 확인된 기록이다. 데이터가 없을 때 아래 셀은 이 숫자를 재생성하지 않고 필요한 입력을 안내한다.

## 2. 입력 구조와 경로 설정

CP v1/v2는 압축파일이며 과거 실험 기록 확인에만 사용한다. Notebook은 이를 읽기 전용으로 확인하고 압축 해제하거나 수정하지 않는다. CP v3~v5만 다음 폴더 구조를 사용한다.

```text
dataset_root/
├── images/*.png
├── annotations/*.json
├── generation_manifest.csv       # 존재하는 경우
├── object_manifest.csv           # 존재하는 경우
└── validation_summary.json       # 존재하는 경우
```

대용량 데이터는 repository 밖에 둘 수 있다. 환경변수 `CP_V1_PATH`~`CP_V5_PATH` 또는 아래 dictionary를 사용한다. v1/v2 환경변수에는 ZIP 파일, v3~v5에는 데이터셋 폴더를 지정한다.

In [ ]:
from pathlib import Path
import csv
import json
import os
import subprocess
import sys
from zipfile import BadZipFile, ZipFile

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
VALIDATOR = PROJECT_ROOT / 'src' / 'data_preparation' / 'validate_copy_paste_quality.py'

DATASETS = {
    'cp_v1': {'kind': 'archive', 'path': Path(os.environ.get('CP_V1_PATH', PROJECT_ROOT / 'data' / 'copy_paste_v1.zip'))},
    'cp_v2': {'kind': 'archive', 'path': Path(os.environ.get('CP_V2_PATH', PROJECT_ROOT / 'data' / 'copy_paste_v2.zip'))},
    'cp_v3': {'kind': 'directory', 'path': Path(os.environ.get('CP_V3_PATH', PROJECT_ROOT / 'data' / 'copy_paste_v3'))},  # CP63
    'cp_v4': {'kind': 'directory', 'path': Path(os.environ.get('CP_V4_PATH', PROJECT_ROOT / 'data' / 'copy_paste_v4'))},  # CP126
    'cp_v5': {'kind': 'directory', 'path': Path(os.environ.get('CP_V5_PATH', PROJECT_ROOT / 'data' / 'copy_paste_v5'))},  # CP500 final
}

assert VALIDATOR.is_file(), f'검증 스크립트를 찾을 수 없습니다: {VALIDATOR}'
DATASETS

## 3. 단계별 개선 근거

| 단계 | 확인된 문제 또는 목적 | 실제 개선·검증 기준 |
|---|---|---|
| CP v1 | 초기 합성 결과 | 압축파일을 과거 실험 기록으로 보존 |
| CP v2 | 육안 검수에서 겹침과 비정상적인 크기·배치 사례 확인 | 압축파일을 후속 개선의 비교 기준으로 보존 |
| CP v3 (CP63) | TS 객체 크기와 실제 Train 객체 분포 차이 | 동일 category의 Train bbox shape에 맞춘 deterministic isotropic scaling, `use` 객체만 사용 |
| CP v4 (CP126) | CP63 유지 및 신규 이미지 확장 | CP63 byte 보존, image/JSON pair, ID, bbox, overlap, 이미지 중복 검증 |
| CP v5 (CP500 final) | 확대 객체, 겹침, 밀집, 경계 및 비정상 배경 사례 | bbox overlap 금지, 최소 GAP 20 px, 경계 검사, category별 bbox area IQR 검사, pair 검증 |

3개 객체는 삼각형 기반, 4개 객체는 사각형 기반 불규칙 배치를 사용했다. 이 Notebook은 생성 로직을 다시 구현하지 않고 최종 산출물 검증에 집중한다.

## 4. 기존 manifest와 validation 결과 확인

다음 셀은 파일이 실제로 존재할 때만 기록을 읽는다. 파일이 없으면 필요한 경로를 출력하며, 과거 수치를 새로 계산한 결과처럼 표시하지 않는다.

In [ ]:
def inspect_recorded_results(label: str, source: dict) -> dict:
    root = source['path']
    result = {'dataset': label, 'kind': source['kind'], 'path': str(root), 'available': root.exists()}
    if source['kind'] == 'archive':
        if not root.is_file():
            result['message'] = '압축파일이 없습니다. CP_V1_PATH 또는 CP_V2_PATH를 지정하세요.'
            return result
        try:
            with ZipFile(root) as archive:
                names = [item.filename for item in archive.infolist() if not item.is_dir()]
            result.update({'archive_files': len(names), 'png_files': sum(name.lower().endswith('.png') for name in names), 'json_files': sum(name.lower().endswith('.json') for name in names)})
        except BadZipFile as exc:
            result.update({'available': False, 'message': f'유효한 ZIP 파일이 아닙니다: {exc}'})
        return result
    if not root.is_dir():
        result['message'] = '입력 폴더가 없습니다. CP_V3_PATH~CP_V5_PATH를 지정하세요.'
        return result

    for name in ('generation_manifest.csv', 'object_manifest.csv'):
        path = root / name
        if path.is_file():
            with path.open(encoding='utf-8-sig', newline='') as handle:
                result[f'{name}_rows'] = sum(1 for _ in csv.DictReader(handle))
    summary_path = root / 'validation_summary.json'
    if summary_path.is_file():
        result['validation_summary'] = json.loads(summary_path.read_text(encoding='utf-8'))
    return result

recorded_results = [inspect_recorded_results(label, source) for label, source in DATASETS.items()]
recorded_results

## 5. 공통 품질 validation 실행

Repository 검증기는 CP v3(CP63), CP v4(CP126), CP v5(CP500 final) 코드에서 공통으로 재현 가능한 항목만 검사한다. CP v1/v2 ZIP은 검증 대상으로 자동 전달하지 않는다.

- PNG/JSON basename 1:1 pair
- JSON의 image record 및 annotation 객체 수
- bbox 좌표와 이미지 경계
- bbox overlap
- 객체 bbox 사이 최소 간격 20 px
- category별 bbox area의 Tukey 1.5 IQR 이상치

이 검사는 읽기 전용이며 `--report`를 명시한 경우에만 새 JSON 보고서를 쓴다.

In [ ]:
def run_validation(label: str, root: Path, expected_images: int | None = None) -> dict:
    if not (root / 'images').is_dir() or not (root / 'annotations').is_dir():
        return {
            'dataset': label,
            'skipped': True,
            'message': f'{root} 아래 images/와 annotations/를 준비하세요.',
        }
    command = [sys.executable, str(VALIDATOR), str(root), '--min-gap', '20']
    if expected_images is not None:
        command.extend(['--expected-images', str(expected_images)])
    completed = subprocess.run(command, text=True, capture_output=True, check=False)
    if not completed.stdout.strip():
        raise RuntimeError(completed.stderr.strip() or f'{label} validation failed without output')
    result = json.loads(completed.stdout)
    result['dataset'] = label
    result['return_code'] = completed.returncode
    return result

expected_counts = {'cp_v3': 63, 'cp_v4': 126, 'cp_v5': 500}
validation_results = [
    run_validation(label, source['path'], expected_counts[label])
    for label, source in DATASETS.items()
    if source['kind'] == 'directory'
]
validation_results

## 6. 최종 확인 항목

검증 결과에서 다음을 확인한다.

1. `status == 'pass'`
2. `missing_annotation_files`와 `missing_image_files`가 비어 있음
3. `bbox_overlap`, `minimum_gap`, `boundary`가 0건
4. `annotation_object_count_distribution`이 의도한 3개·4개 구성과 일치
5. `category_size_outlier`가 있으면 원본 Train 분포 및 객체 manifest와 함께 육안 review

크기 이상치는 자동 삭제 기준이 아니라 CP v5(CP500 final) 보정 과정에서 사용한 검토 신호다. 원본 Train 데이터, 대용량 PNG, processed dataset, TAR, model weight와 review 이미지는 Git에 포함하지 않는다.